# 🏥 Healthcare FL: Phase 2 Regulatory Verification (Colab Pro)

**Audit Scope:** Environment Reproducibility, DP Serialization, and Model Convergence.

---

## 1. Environment Synchronization
Mount Drive, unzip project, and bootstrap dependencies.

In [ ]:
from google.colab import drive
import os
import sys

drive.mount('/content/drive')

# CONFIG: Adjust these paths if your zip or target folder is named differently
ZIP_FILE = '/content/drive/MyDrive/Healthcare_FL_Verification.zip'
TARGET_DIR = '/content/healthcare_project'

if not os.path.exists(TARGET_DIR):
    os.makedirs(TARGET_DIR)
    print(f"Created directory {TARGET_DIR}")

print(f"Unzipping {ZIP_FILE} to {TARGET_DIR}...")
!unzip -q {ZIP_FILE} -d {TARGET_DIR}

%cd {TARGET_DIR}

!python scripts/colab_bootstrap.py

## 2. Regulatory Environment Audit
Profile system state and resources.

In [ ]:
import torch
import os
import platform
import psutil

print(f"--- 🛡️ Regulatory Audit: System State ---")
print(f"OS: {platform.platform()}")
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\n--- 🖥️ Resource Profile ---")
cpus = os.cpu_count()
ram = psutil.virtual_memory().total / (1024**3)
print(f"CPUs: {cpus}")
print(f"RAM: {ram:.2f} GB")

# Resource Guard
assert cpus >= 2, "❌ Critical: Minimum 2 CPUs required for Ray simulation."
print("✅ Resource requirements met.")

## 3. DP State Serialization Validation
Prove that Differential Privacy states persist across client re-instantiations.

In [ ]:
from src.federated.client import FlowerClient
import torch
import os
import pickle

print("--- 🧪 DP Persistence Test: Round-Trip Serialization ---")

# 1. Initialize Client & Perform Step
client = FlowerClient(center_id=0, dry_run=True)
initial_eps = client.privacy_engine.get_epsilon(delta=1e-5)
print(f"Initial Epsilon: {initial_eps}")

# Run one fit step (simulated)
weights = client.get_parameters(config={})
_, _, _ = client.fit(weights, config={"lr": 0.001})
step_1_eps = client.privacy_engine.get_epsilon(delta=1e-5)
print(f"Epsilon after 1 step: {step_1_eps}")

# 2. Verify File Creation
state_path = f"ray_tmp/dp_state_client_0.pkl"
assert os.path.exists(state_path), "❌ Error: DP state file not created!"
print(f"✅ State file persisted to {state_path}")

# 3. Re-instantiate and Verify Reload
new_client = FlowerClient(center_id=0, dry_run=True)
reloaded_eps = new_client.privacy_engine.get_epsilon(delta=1e-5)
print(f"Reloaded Epsilon: {reloaded_eps}")

assert abs(reloaded_eps - step_1_eps) < 1e-7, "❌ Error: DP State Mismatch on Reload!"
print("✅ DP Persistence Proof: PASSED")

## 4. Federated Simulation & Convergence Audit
Execute the simulation and monitor for diagnostic accuracy vs. privacy budget trade-offs.

In [ ]:
print("--- 🏥 Verification: 2-Round Dry-Run ---")
!python src/federated/server.py --num-rounds 2 --dry-run --no-mlflow

print("\n--- 🚀 Final Gate: 3-Round Full Simulation ---")
!python src/federated/server.py --num-rounds 3 --no-mlflow

## 5. Evidence Extraction
Package all logs and artifacts for regulatory submission.

In [ ]:
!zip -r Verification_Artifacts_Phase2.zip ray_tmp/ eda_outputs/ notebooks/Phase_2_Colab_Verification.ipynb
print("✅ Evidence package created: Verification_Artifacts_Phase2.zip")